In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
# ================================================================
# PHASE 25 — COPILOT API
# FILE: 08_integration/06_copilot_api.py
# ================================================================

print("=" * 70)
print("PHASE 25 — COPILOT API")
print("=" * 70)

print("Building production API interface for GenAI Data Analyst Copilot")

In [0]:
# ================================================================
# CELL 2 — LOAD COPILOT ORCHESTRATOR
# ================================================================

print("=" * 70)
print("LOADING COPILOT ORCHESTRATOR")
print("=" * 70)


print(
    "ask_copilot:",
    "PASS"
    if callable(globals().get("ask_copilot"))
    else "FAIL"
)

if not callable(
    globals().get("ask_copilot")
):

    raise RuntimeError(
        "ask_copilot is not available."
    )

print()
print("Copilot orchestrator: PASS")

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/03_production_response

In [0]:
# ================================================================
# CELL 3 — LOAD PRODUCTION RESPONSE FORMATTER
# ================================================================

print("=" * 70)
print("LOADING PRODUCTION RESPONSE FORMATTER")
print("=" * 70)



print(
    "format_copilot_response:",
    "PASS"
    if callable(
        globals().get(
            "format_copilot_response"
        )
    )
    else "FAIL"
)

if not callable(
    globals().get(
        "format_copilot_response"
    )
):

    raise RuntimeError(
        "format_copilot_response is not available."
    )

print()
print("Production response formatter: PASS")

In [0]:
# ================================================================
# CELL 4 — API DEPENDENCY VALIDATION
# ================================================================

print("=" * 70)
print("API DEPENDENCY VALIDATION")
print("=" * 70)

required_functions = [
    "ask_copilot",
    "format_copilot_response"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(
            function_name
        )
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:

        failed_dependencies.append(
            function_name
        )

print()

print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "Missing dependencies: "
        + ", ".join(
            failed_dependencies
        )
    )

print()
print("API dependency validation: PASS")

In [0]:
# ================================================================
# CELL 5 — API REQUEST VALIDATION
# ================================================================

print("=" * 70)
print("API REQUEST VALIDATION")
print("=" * 70)


def validate_api_request(request):

    if request is None:

        return {
            "valid": False,
            "error": "Request cannot be null."
        }

    if not isinstance(
        request,
        dict
    ):

        return {
            "valid": False,
            "error": "Request must be a JSON object."
        }

    if "question" not in request:

        return {
            "valid": False,
            "error": "Missing required field: question."
        }

    question = request.get(
        "question"
    )

    if not isinstance(
        question,
        str
    ):

        return {
            "valid": False,
            "error": "question must be a string."
        }

    question = question.strip()

    if not question:

        return {
            "valid": False,
            "error": "question cannot be empty."
        }

    return {
        "valid": True,
        "question": question,
        "error": None
    }


print(
    "validate_api_request: PASS"
)

In [0]:
# ================================================================
# CELL 6 — API RESPONSE BUILDER
# ================================================================

print("=" * 70)
print("API RESPONSE BUILDER")
print("=" * 70)


def build_api_response(
    request
):

    validation = validate_api_request(
        request
    )

    if not validation["valid"]:

        return {
            "success": False,
            "question": request.get(
                "question"
            )
            if isinstance(
                request,
                dict
            )
            else None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": validation["error"]
        }

    question = validation[
        "question"
    ]

    try:

        result = ask_copilot(
            question
        )

        formatted_response = (
            format_copilot_response(
                result
            )
        )

        return {
            "success": result.get(
                "success",
                True
            ),
            "question": question,
            "route": result.get(
                "route"
            ),
            "answer": formatted_response,
            "sql": result.get(
                "sql"
            ),
            "data": result.get(
                "data"
            ),
            "sources": result.get(
                "sources",
                []
            ),
            "error": result.get(
                "error"
            )
        }

    except Exception as e:

        return {
            "success": False,
            "question": question,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                f"{type(e).__name__}: "
                f"{str(e)}"
            )
        }


print(
    "build_api_response: PASS"
)

In [0]:
# ================================================================
# CELL 7 — SQL API TEST
# ================================================================

print("=" * 70)
print("SQL API TEST")
print("=" * 70)

sql_request = {
    "question":
        "Which region generated the highest revenue?"
}

sql_api_response = build_api_response(
    sql_request
)

print(
    json.dumps(
        sql_api_response,
        indent=2,
        default=str
    )
)

if not sql_api_response["success"]:

    raise RuntimeError(
        "SQL API test failed: "
        + str(
            sql_api_response["error"]
        )
    )

if sql_api_response["route"] != "sql":

    raise RuntimeError(
        "Expected SQL route but received: "
        + str(
            sql_api_response["route"]
        )
    )

print()
print("SQL API test: PASS")

In [0]:
# ================================================================
# CELL 8 — RAG API TEST
# ================================================================

print("=" * 70)
print("RAG API TEST")
print("=" * 70)

rag_request = {
    "question":
        "What is the discount policy?"
}

rag_api_response = build_api_response(
    rag_request
)

print(
    json.dumps(
        rag_api_response,
        indent=2,
        default=str
    )
)

if not rag_api_response["success"]:

    raise RuntimeError(
        "RAG API test failed: "
        + str(
            rag_api_response["error"]
        )
    )

if rag_api_response["route"] != "rag":

    raise RuntimeError(
        "Expected RAG route but received: "
        + str(
            rag_api_response["route"]
        )
    )

print()
print("RAG API test: PASS")

In [0]:
# ================================================================
# CELL 9 — HYBRID API TEST
# ================================================================

print("=" * 70)
print("HYBRID API TEST")
print("=" * 70)

hybrid_request = {
    "question": (
        "Which region generated the highest revenue "
        "and what discount policy applies there?"
    )
}

hybrid_api_response = build_api_response(
    hybrid_request
)

print(
    json.dumps(
        hybrid_api_response,
        indent=2,
        default=str
    )
)

if not hybrid_api_response["success"]:

    raise RuntimeError(
        "Hybrid API test failed: "
        + str(
            hybrid_api_response["error"]
        )
    )

if hybrid_api_response["route"] != "hybrid":

    raise RuntimeError(
        "Expected Hybrid route but received: "
        + str(
            hybrid_api_response["route"]
        )
    )

print()
print("Hybrid API test: PASS")

In [0]:
# ================================================================
# CELL 10 — INVALID REQUEST TESTS
# ================================================================

print("=" * 70)
print("INVALID REQUEST TESTS")
print("=" * 70)

invalid_requests = [
    None,
    {},
    {"question": ""},
    {"question": "   "},
    {"question": 123}
]

invalid_passed = 0

for request in invalid_requests:

    response = build_api_response(
        request
    )

    passed = (
        response["success"] is False
        and response["error"] is not None
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{request}"
    )

    if passed:

        invalid_passed += 1

print()

print(
    "Invalid requests tested:",
    len(invalid_requests)
)

print(
    "Passed:",
    invalid_passed
)

if invalid_passed != len(
    invalid_requests
):

    raise RuntimeError(
        "Invalid request validation failed."
    )

print()
print("Invalid request validation: PASS")

In [0]:
# ================================================================
# CELL 11 — API CONTRACT VALIDATION
# ================================================================

print("=" * 70)
print("API CONTRACT VALIDATION")
print("=" * 70)

required_response_fields = [
    "success",
    "question",
    "route",
    "answer",
    "sql",
    "data",
    "sources",
    "error"
]

response_to_validate = (
    hybrid_api_response
)

missing_fields = [
    field
    for field in required_response_fields
    if field not in response_to_validate
]

for field in required_response_fields:

    print(
        f"{'PASS' if field not in missing_fields else 'FAIL'} - "
        f"{field}"
    )

print()

if missing_fields:

    raise RuntimeError(
        "Missing API response fields: "
        + ", ".join(
            missing_fields
        )
    )

print(
    "API contract validation: PASS"
)

In [0]:
# ================================================================
# CELL 12 — FINAL API VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 25 — COPILOT API VALIDATION")
print("=" * 70)

api_tests = {
    "SQL": sql_api_response["success"]
    and sql_api_response["route"] == "sql",

    "RAG": rag_api_response["success"]
    and rag_api_response["route"] == "rag",

    "HYBRID": hybrid_api_response["success"]
    and hybrid_api_response["route"] == "hybrid",

    "INVALID_REQUESTS":
        invalid_passed
        == len(invalid_requests),

    "API_CONTRACT":
        len(missing_fields) == 0
}

successful_tests = sum(
    1
    for value in api_tests.values()
    if value
)

failed_tests = (
    len(api_tests)
    - successful_tests
)

for test_name, passed in api_tests.items():

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{test_name}"
    )

print()

print(
    "Total API tests:",
    len(api_tests)
)

print(
    "Successful API tests:",
    successful_tests
)

print(
    "Failed API tests:",
    failed_tests
)

print()

if failed_tests:

    print(
        "PHASE 25 STATUS: FAIL ✗"
    )

    raise RuntimeError(
        "Copilot API validation failed."
    )

print(
    "PHASE 25 STATUS: PASS ✓"
)

In [0]:
# ================================================================
# CELL 13 — EXAMPLE API PAYLOAD
# ================================================================

example_request = {
    "question":
        "Which region generated the highest revenue?"
}

print(
    json.dumps(
        example_request,
        indent=2
    )
)

print()
print(
    "This JSON structure will be used by the "
    "future Hugging Face Gradio frontend."
)